### Data Description and Preparation

The electricity consumption data is originally provided at a 30-minute resolution for each French region. To combine it with the weather dataset, which is only available at a daily frequency, we aggregate electricity consumption to the daily level.

For each region and each day, all 30-minute consumption values are summed to obtain the total daily electricity demand. The resulting daily dataset is then merged with the daily weather data using the common date and region identifiers.

This ensures both datasets are aligned at the same temporal resolution before descriptive analysis and modeling.

In [1]:
pip install -q cartiflette

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px

In [3]:
# ele = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/RTE/data_ele-2020-2024.csv")
# climate = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/weather/Region_France_weather_daily_2020_2024.csv")

df = pd.read_csv("/home/onyxia/work/Python_Project_2A/data/temp_electricity.csv")
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [4]:
# ele

In [5]:
# climate

1. **Does electricity consumption vary systematically with temperature?**  

In [6]:
df

,Datetime,Consommation,Regions,Nature,year,month,day,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,GWETROOT,...,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,WD10M,WS10M,WS2M,insee_dep
0,2020-01-01,403787.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,1,5.39,6.80,0.68,...,4.17,96.11,0.81,5.26,-1.36,6.62,314.6,1.44,0.82,83
1,2020-01-02,443531.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,2,4.78,6.45,0.68,...,4.41,92.78,2.44,8.35,-0.89,9.24,200.2,3.64,2.26,83
2,2020-01-03,434626.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,3,3.80,6.38,0.68,...,5.00,93.72,3.94,8.16,1.49,6.67,246.6,3.21,1.91,83
3,2020-01-04,395169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,4,3.91,6.99,0.68,...,4.76,94.68,3.18,5.62,-0.64,6.26,335.6,4.34,2.77,83
4,2020-01-05,400169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,5,5.85,7.13,0.67,...,3.96,93.92,0.77,4.06,-1.25,5.31,354.9,3.75,2.43,83
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21919,2024-12-27,181429.0,Pays-de-la-Loire,Données consolidées,2024,12,27,2.05,5.86,0.55,...,3.91,94.54,1.20,5.31,-3.47,8.78,115.8,1.72,0.97,52
21920,2024-12-28,171307.0,Pays-de-la-Loire,Données consolidées,2024,12,28,2.10,5.59,0.55,...,4.93,98.06,4.17,5.52,3.20,2.32,188.7,1.12,0.68,52
21921,2024-12-29,169980.0,Pays-de-la-Loire,Données consolidées,2024,12,29,2.32,5.64,0.55,...,4.17,93.09,2.60,4.63,0.94,3.69,72.1,1.22,0.80,52
21922,2024-12-30,185431.0,Pays-de-la-Loire,Données consolidées,2024,12,30,2.34,5.69,0.55,...,3.90,95.37,1.30,2.89,-0.17,3.06,172.1,1.81,1.14,52


In [7]:
df.describe()

,Datetime,Consommation,year,month,day,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,GWETROOT,GWETTOP,PRECTOTCORR,...,QV2M,RH2M,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,WD10M,WS10M,WS2M,insee_dep
count,21924,21924.000000,21924.00000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,...,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000,21924.000000
mean,2022-07-02 00:00:00,204433.967843,2022.00000,6.521073,15.735085,12.832862,17.736459,0.575099,0.596883,1.994065,...,7.198033,79.303629,12.013618,16.923766,7.537540,9.386226,195.470799,4.279693,2.812103,49.833333
min,2020-01-01 00:00:00,58376.000000,2020.00000,1.000000,1.000000,0.460000,3.730000,0.220000,0.100000,0.000000,...,1.700000,23.680000,-8.040000,-3.690000,-12.570000,0.680000,0.000000,0.620000,0.400000,11.000000
25%,2021-04-01 00:00:00,125892.500000,2021.00000,4.000000,8.000000,5.710000,9.320000,0.480000,0.500000,0.030000,...,5.320000,71.980000,6.827500,10.920000,3.100000,6.700000,114.200000,2.820000,1.770000,27.750000
50%,2022-07-02 00:00:00,191000.500000,2022.00000,7.000000,16.000000,11.700000,17.800000,0.570000,0.610000,0.360000,...,6.990000,81.770000,11.635000,16.440000,7.570000,9.100000,213.300000,3.890000,2.530000,48.000000
75%,2023-10-02 00:00:00,262712.500000,2023.00000,10.000000,23.000000,19.062500,26.110000,0.660000,0.700000,2.350000,...,8.940000,89.630000,17.380000,22.710000,12.170000,11.930000,274.300000,5.340000,3.560000,75.250000
max,2024-12-31 00:00:00,624804.000000,2024.00000,12.000000,31.000000,32.360000,32.990000,0.960000,0.940000,53.790000,...,16.360000,100.000000,33.310000,42.330000,24.850000,21.630000,360.000000,15.030000,10.600000,93.000000
std,NaN,94293.008281,1.41502,3.449291,8.802592,7.962586,8.521331,0.129210,0.138613,3.699876,...,2.469397,13.039571,6.930725,7.922475,6.030541,3.566004,100.094157,1.967521,1.393298,25.531065


In [8]:
df_total = df.groupby("Datetime", as_index=False)["Consommation"].sum()

fig = px.line(df_total, x="Datetime", y="Consommation",
              title="Total Electricity Consumption in France Over Time"
              ,labels={"Consommation": "Consommation (MWh)"})
fig.show()

**Observation :**

In [9]:
from cartiflette import carti_download
import geopandas as gpd
import matplotlib.pyplot as plt
import json

In [10]:
france = carti_download(
      values = ["France"],
      crs = 4326,
      borders = "REGION",
      vectorfile_format="geojson",
      simplification=50,
      filter_by="FRANCE_ENTIERE",
      source="EXPRESS-COG-CARTO-TERRITOIRE",
      year=2022)
france = france.loc[(france["INSEE_REG"] > 10) & (france["INSEE_REG"] != 94)]

france

,INSEE_REG,PAYS,LIBELLE_REGION,POPULATION,SOURCE,geometry
4,84,France,Auvergne-Rhône-Alpes,8042936,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((6.06386 46.41639, 6.06267 46.4168, 6..."
5,76,France,Occitanie,5933185,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((2.06288 44.97662, 2.06244 44.9..."
6,53,France,Bretagne,3354854,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-3.97907 47.70396, -3.97953 47..."
7,75,France,Nouvelle-Aquitaine,6010289,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-0.31325 42.8494, -0.31227 42...."
8,28,France,Normandie,3325032,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-1.924 49.66512, -1.92439 49.6..."
9,93,France,Provence-Alpes-Côte d'Azur,5081101,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((4.23021 43.46049, 4.2329 43.46..."
10,52,France,Pays de la Loire,3806461,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"MULTIPOLYGON (((-1.10454 46.31496, -1.10269 46..."
11,44,France,Grand Est,5556219,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((3.41474 48.39019, 3.41443 48.38781, ..."
12,27,France,Bourgogne-Franche-Comté,2805580,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((7.13026 47.50295, 7.13157 47.50344, ..."
13,11,France,Île-de-France,12262544,IGN:EXPRESS-COG-CARTO-TERRITOIRE,"POLYGON ((1.5014 48.94103, 1.51182 48.93499, 1..."


In [11]:
france.columns

Index(['INSEE_REG', 'PAYS', 'LIBELLE_REGION', 'POPULATION', 'SOURCE',
       'geometry'],
      dtype='object')

In [12]:
# Now i have 2 data which is the my df data and the other one is the regions_gdf. So in order to plot the map of the consumption each region using my data, i need to make sure if the region of my data match with the regions_gdf.

In [13]:
df["Regions"].nunique()
# df["Regions"].unique()

12

In [14]:
# sorted(df["Regions"].unique())
# sorted(france["LIBELLE_REGION"].unique())

In [15]:
#check if the region match to the regions_gdf data. It does exist but just different spelling so need to correct the spelling in df data

set(df["Regions"].unique()) - set(france["LIBELLE_REGION"].unique())

{'Grand-Est', 'Ile-de-France', 'PACA', 'Pays-de-la-Loire'}

In [16]:
#correct spelling

region_mapping = {
    "Grand-Est": "Grand Est",
    "Ile-de-France": "Île-de-France",
    "Pays-de-la-Loire": "Pays de la Loire",
    "PACA": "Provence-Alpes-Côte d'Azur"
}

df["Regions_clean"] = df["Regions"].replace(region_mapping)

In [17]:
set(france["LIBELLE_REGION"].unique()) - set(df["Regions_clean"].unique())

set()

In [18]:
france_geojson = json.loads(france.to_json())
france_geojson

{'type': 'FeatureCollection',
 'features': [{'id': '4',
   'type': 'Feature',
   'properties': {'INSEE_REG': 84,
    'PAYS': 'France',
    'LIBELLE_REGION': 'Auvergne-Rhône-Alpes',
    'POPULATION': 8042936,
    'SOURCE': 'IGN:EXPRESS-COG-CARTO-TERRITOIRE'},
   'geometry': {'type': 'Polygon',
    'coordinates': [[[6.063857350782006, 46.416389647447375],
      [6.0626684922207925, 46.41679733645055],
      [6.060147193152213, 46.41791119839804],
      [6.059644440229763, 46.41744977945653],
      [6.0585931776113595, 46.41724456631177],
      [6.057640555619242, 46.41668345495181],
      [6.0570186773400065, 46.416560440061],
      [6.05564508429009, 46.415647431628756],
      [6.0545916377043, 46.41463392238324],
      [6.054329634336186, 46.414166036966435],
      [6.053227050250644, 46.413134907753395],
      [6.053009458788276, 46.41192329552913],
      [6.052056354930915, 46.41125762235352],
      [6.051680771258537, 46.410644983513684],
      [6.051907693090389, 46.40971705259307]

In [19]:
# print("Number of features:", len(france_geojson["features"]))
# print([f["properties"]["LIBELLE_REGION"] for f in france_geojson["features"]])

In [20]:
df

,Datetime,Consommation,Regions,Nature,year,month,day,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,GWETROOT,...,RH2M,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,WD10M,WS10M,WS2M,insee_dep,Regions_clean
0,2020-01-01,403787.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,1,5.39,6.80,0.68,...,96.11,0.81,5.26,-1.36,6.62,314.6,1.44,0.82,83,Auvergne-Rhône-Alpes
1,2020-01-02,443531.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,2,4.78,6.45,0.68,...,92.78,2.44,8.35,-0.89,9.24,200.2,3.64,2.26,83,Auvergne-Rhône-Alpes
2,2020-01-03,434626.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,3,3.80,6.38,0.68,...,93.72,3.94,8.16,1.49,6.67,246.6,3.21,1.91,83,Auvergne-Rhône-Alpes
3,2020-01-04,395169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,4,3.91,6.99,0.68,...,94.68,3.18,5.62,-0.64,6.26,335.6,4.34,2.77,83,Auvergne-Rhône-Alpes
4,2020-01-05,400169.0,Auvergne-Rhône-Alpes,Données définitives,2020,1,5,5.85,7.13,0.67,...,93.92,0.77,4.06,-1.25,5.31,354.9,3.75,2.43,83,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21919,2024-12-27,181429.0,Pays-de-la-Loire,Données consolidées,2024,12,27,2.05,5.86,0.55,...,94.54,1.20,5.31,-3.47,8.78,115.8,1.72,0.97,52,Pays de la Loire
21920,2024-12-28,171307.0,Pays-de-la-Loire,Données consolidées,2024,12,28,2.10,5.59,0.55,...,98.06,4.17,5.52,3.20,2.32,188.7,1.12,0.68,52,Pays de la Loire
21921,2024-12-29,169980.0,Pays-de-la-Loire,Données consolidées,2024,12,29,2.32,5.64,0.55,...,93.09,2.60,4.63,0.94,3.69,72.1,1.22,0.80,52,Pays de la Loire
21922,2024-12-30,185431.0,Pays-de-la-Loire,Données consolidées,2024,12,30,2.34,5.69,0.55,...,95.37,1.30,2.89,-0.17,3.06,172.1,1.81,1.14,52,Pays de la Loire


In [21]:
# Daily consumption per region (from cleaned names)
df_day_region = (
    df.groupby(["Datetime", "Regions_clean"], as_index=False)["Consommation"]
      .sum()
)

# Create Month as a period
df_day_region["Month"] = df_day_region["Datetime"].dt.to_period("M")

# Monthly consumption per region
df_month_region = (
    df_day_region
    .groupby(["Month", "Regions_clean"], as_index=False)["Consommation"]
    .sum()
)

# Convert Month back to timestamp and make labels
df_month_region["Month"] = df_month_region["Month"].dt.to_timestamp()
df_month_region["year"] = df_month_region["Month"].dt.year
df_month_region["month_str"] = df_month_region["Month"].dt.strftime("%Y-%m")

df_month_region

,Month,Regions_clean,Consommation,year,month_str
0,2020-01-01,Auvergne-Rhône-Alpes,13645505.0,2020,2020-01
1,2020-01-01,Bourgogne-Franche-Comté,4388881.0,2020,2020-01
2,2020-01-01,Bretagne,4910735.0,2020,2020-01
3,2020-01-01,Centre-Val de Loire,4094049.0,2020,2020-01
4,2020-01-01,Grand Est,9123842.0,2020,2020-01
...,...,...,...,...,...
715,2024-12-01,Nouvelle-Aquitaine,8678148.0,2024,2024-12
716,2024-12-01,Occitanie,7804218.0,2024,2024-12
717,2024-12-01,Pays de la Loire,5323098.0,2024,2024-12
718,2024-12-01,Provence-Alpes-Côte d'Azur,7969958.0,2024,2024-12


In [22]:
year_to_plot = 2022   # <- change this to 2020, 2021, 2023, etc.

df_month_year = df_month_region[df_month_region["year"] == year_to_plot]

# fig = px.choropleth_mapbox(
#     df_month_year,
#     geojson=france_geojson,
#     locations="Regions_clean",                 # column in df_month_year
#     featureidkey="properties.LIBELLE_REGION",  # property in geojson
#     color="Consommation",
#     animation_frame="month_str",               # monthly slider
#     mapbox_style="carto-positron",
#     center={"lat": 46.5, "lon": 2.5},
#     zoom=4.5,
#     opacity=0.7,
#     color_continuous_scale="Viridis",
#     labels={
#         "Consommation": "Consumption (MWh)",
#         "month_str": "Month"
#     },
#     title=f"Monthly Electricity Consumption by Region in France – {year_to_plot}"
# )

# # Add borders so regions are visually clear
# fig.update_traces(marker_line_width=0.5, marker_line_color="black")

# fig.update_layout(
#     margin={"r":0, "t":40, "l":0, "b":0}
# )

# fig.show()